In [ ]:
# ── 1. Clone your repo ──
!git clone https://github.com/Anand-786/llm-quantization-thesis.git
%cd /content/llm-quantization-thesis

# ── 2. Clone SmoothQuant repo and install it properly ──
!git clone https://github.com/mit-han-lab/smoothquant.git smoothquant_repo
!pip uninstall smoothquant -y
!cd smoothquant_repo && pip install -e .

# ── 3. Install other dependencies ──
!pip install -q transformers accelerate datasets zstandard tqdm

# ── 4. Download the Pile validation set (calibration data) ──
!mkdir -p smoothquant_repo/dataset
!wget -O smoothquant_repo/dataset/val.jsonl.zst https://huggingface.co/datasets/mit-han-lab/pile-val-backup/resolve/main/val.jsonl.zst

# ── 5. Mount Drive ──
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/thesis_results/task01

# ── 6. Verify ──
!nvidia-smi
!python -c "from smoothquant.smooth import smooth_lm; print('✅ smoothquant.smooth imports OK')"
!python -c "from smoothquant.fake_quant import quantize_model; print('✅ smoothquant.fake_quant imports OK')"
!ls -la smoothquant_repo/dataset/

Cloning into 'llm-quantization-thesis'...
remote: Enumerating objects: 49, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 49 (delta 11), reused 46 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (49/49), 27.06 KiB | 6.76 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/llm-quantization-thesis
Cloning into 'smoothquant_repo'...
remote: Enumerating objects: 352, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 352 (delta 120), reused 90 (delta 90), pack-reused 183 (from 1)
Receiving objects: 100% (352/352), 6.80 MiB | 22.31 MiB/s, done.
Resolving deltas: 100% (202/202), done.
Obtaining file:///content/llm-quantization-thesis/smoothquant_repo
  Preparing metadata (setup.py) ... done
  Running setup.py develop for smoothquant
--2026-03-22 23:48:10--  https://huggingface.co/datasets/mit-han-lab/pile-val-backup/resolve/main/val.jsonl.zst

In [ ]:
%cd /content/llm-quantization-thesis/smoothquant_repo

# OPT-125m (tiny, fast, good for testing)
!python examples/generate_act_scales.py \
    --model-name facebook/opt-125m \
    --output-path act_scales/opt-125m.pt \
    --dataset-path dataset/val.jsonl.zst \
    --num-samples 512 \
    --seq-len 512

# OPT-1.3b
!python examples/generate_act_scales.py \
    --model-name facebook/opt-1.3b \
    --output-path act_scales/opt-1.3b.pt \
    --dataset-path dataset/val.jsonl.zst \
    --num-samples 512 \
    --seq-len 512

# Verify scales were created
!ls -la act_scales/

# Backup to Drive so you never have to regenerate
!cp act_scales/*.pt /content/drive/MyDrive/thesis_results/act_scales/ 2>/dev/null; \
 mkdir -p /content/drive/MyDrive/thesis_results/act_scales && \
 cp act_scales/*.pt /content/drive/MyDrive/thesis_results/act_scales/

/content/llm-quantization-thesis/smoothquant_repo
config.json: 100% 651/651 [00:00<00:00, 3.23MB/s]
tokenizer_config.json: 100% 685/685 [00:00<00:00, 3.94MB/s]
vocab.json: 899kB [00:00, 111MB/s]
merges.txt: 456kB [00:00, 101MB/s]
special_tokens_map.json: 100% 441/441 [00:00<00:00, 2.65MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
pytorch_model.bin: 100% 251M/251M [00:02<00:00, 91.9MB/s]
Loading weights: 100% 197/197 [00:00<00:00, 1087.28it/s, Materializing param=model.decoder.layers.11.self_attn_layer_norm.weight]
The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
generation_config.json: 100% 137/137 [00:00<00:00, 951kB/s]
Generating train split: 1565 examples [00:00, 11159.39 examples/s]
Generating train split: 7777 examples [00:00, 17369.26 examples/s]
Gen

In [ ]:
%cd /content/llm-quantization-thesis/smoothquant_repo

# Run 1: FP16 baseline (no smooth, no quantize)
!python smoothquant/ppl_eval.py \
    --model_path facebook/opt-125m \
    --act_scales_path act_scales/opt-125m.pt

# Run 2: Quantized WITHOUT smoothing
!python smoothquant/ppl_eval.py \
    --model_path facebook/opt-125m \
    --act_scales_path act_scales/opt-125m.pt \
    --quantize

# Run 3: Smoothed + Quantized (this is SmoothQuant's result)
!python smoothquant/ppl_eval.py \
    --model_path facebook/opt-125m \
    --act_scales_path act_scales/opt-125m.pt \
    --smooth --alpha 0.5 \
    --quantize

/content/llm-quantization-thesis/smoothquant_repo
README.md: 10.5kB [00:00, 18.4MB/s]
wikitext-2-raw-v1/test-00000-of-00001.pa(…): 100% 733k/733k [00:01<00:00, 601kB/s] 
wikitext-2-raw-v1/train-00000-of-00001.p(…): 100% 6.36M/6.36M [00:00<00:00, 10.4MB/s]
wikitext-2-raw-v1/validation-00000-of-00(…): 100% 657k/657k [00:00<00:00, 1.60MB/s]
Generating test split: 100% 4358/4358 [00:00<00:00, 103909.82 examples/s]
Generating train split: 100% 36718/36718 [00:00<00:00, 644049.71 examples/s]
Generating validation split: 100% 3760/3760 [00:00<00:00, 590216.43 examples/s]
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 197/197 [00:00<00:00, 216.08it/s, Materializing param=model.decoder.layers.11.self_attn_layer_norm.weight]
The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` t

In [ ]:
%cd /content/llm-quantization-thesis/smoothquant_repo

# Run 1: FP16 baseline
!python smoothquant/ppl_eval.py \
    --model_path facebook/opt-1.3b \
    --act_scales_path act_scales/opt-1.3b.pt

# Run 2: Quantized WITHOUT smoothing
!python smoothquant/ppl_eval.py \
    --model_path facebook/opt-1.3b \
    --act_scales_path act_scales/opt-1.3b.pt \
    --quantize

# Run 3: Smoothed + Quantized
!python smoothquant/ppl_eval.py \
    --model_path facebook/opt-1.3b \
    --act_scales_path act_scales/opt-1.3b.pt \
    --smooth --alpha 0.5 \
    --quantize

/content/llm-quantization-thesis/smoothquant_repo
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 389/389 [00:09<00:00, 39.70it/s, Materializing param=model.decoder.layers.23.self_attn_layer_norm.weight]
The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Evaluating...: 100% 140/140 [06:30<00:00,  2.79s/it]
Perplexity: 14.627793312072754
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 389/389 [00:03<00:00, 115.30it/s, Materializing param=model.decoder.layers.23.self_attn_layer_norm.weight]
The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embed